In [ ]:
import numpy as np
import pandas as pd
import joblib
from itertools import product
from functools import partial
from multiprocessing import Pool
from tqdm import tqdm
from physics import *

In [ ]:
def process_combination(params, n_layers=1):
    """
    params: [k0, k1, ..., k_{n+1}, h1, h2, ..., hn]
    """
    r_val = forward_model(params=params[:-1], theta=params[-1], n_layers=n_layers)

    result_dict = {
        'theta': params[-1],
        'h': params[n_layers+2:-1],
        'k_up': params[0],
        'k_down': params[n_layers+1],
        'k_vals': params[1:n_layers+1],
        'Re(r)': np.real(r_val),
        'Im(r)': np.imag(r_val)
    }
    return result_dict

def generation_params(k_range, h_range, theta_range, n_layers=1):
    thetas = np.deg2rad(np.arange(theta_range[0], theta_range[1], 10))
    hs = np.arange(h_range[0], h_range[1], 10)
    k_vals = np.linspace(k_range[0], k_range[1], 100)

    params_list = list(product([1], *[k_vals]*n_layers, [1], *[hs]*n_layers, thetas))
    return params_list

In [70]:
params_list = generation_params(k_range=[0.1,1], h_range=[0.1, 3], theta_range=[10, 90], n_layers=2)
params_list[0], len(params_list)

((1, 0.1, 0.1, 1, 0.1, 0.1, 0.17453292519943295), 80000)

In [ ]:
worker = partial(process_combination, n_layers=2)

with Pool() as pool:
    results = list(
        tqdm(pool.imap_unordered(worker, params_list),
             total=len(params_list))
    )

100%|██████████| 80000/80000 [00:35<00:00, 2277.52it/s]


In [68]:
data = pd.DataFrame(results)

In [69]:
data

,theta,h,k_up,k_down,k_vals,Re(r),Im(r)
0,0.698132,"(0.1, 0.1)",1,1,"(0.1, 0.1)",0.003063,-0.128472
1,0.349066,"(0.1, 0.1)",1,1,"(0.1, 0.1)",0.008617,-0.104493
2,0.872665,"(0.1, 0.1)",1,1,"(0.1, 0.1)",-0.003859,-0.152745
3,1.047198,"(0.1, 0.1)",1,1,"(0.1, 0.1)",-0.018850,-0.194240
4,1.221730,"(0.1, 0.1)",1,1,"(0.1, 0.1)",-0.059675,-0.273094
...,...,...,...,...,...,...,...
79995,0.872665,"(0.1, 0.1)",1,1,"(1.0, 1.0)",0.000000,0.000000
79996,1.047198,"(0.1, 0.1)",1,1,"(1.0, 1.0)",0.000000,0.000000
79997,1.221730,"(0.1, 0.1)",1,1,"(1.0, 1.0)",0.000000,0.000000
79998,1.396263,"(0.1, 0.1)",1,1,"(1.0, 1.0)",0.000000,0.000000


In [ ]:
# joblib.dump(data, 'data/one_slay.joblib')

['data/one_slay.joblib']